# Parameter estimation

This notebook demonstrates how to use **marapendi**'s `SteadyStateModel` to fit
model parameters against experimental data.

We cover two use cases:
1. A simple synthetic example (quadratic model) — great for understanding the API.
2. Fitting a PEMFC polarization curve to noisy synthetic measurements.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import marapendi as mrpd

## Part 1 — Synthetic example

Fit the two parameters `a` and `b` of:

$$y(x) = a \, x^2 + b \, x$$

from noisy observations. The true values are `a = 3`, `b = 1.5`.

In [ ]:
rng = np.random.default_rng(42)

X = np.linspace(0., 1., 30)
TRUE_PARAMS = {'a': 3.0, 'b': 1.5}

def model_fn(params):
    """Compute model predictions given a parameter dict."""
    return params['a'] * X**2 + params['b'] * X

y_true = model_fn(TRUE_PARAMS)
y_noisy = y_true + rng.normal(0, 0.02, size=X.shape)

plt.figure(figsize=(5, 3))
plt.scatter(X, y_noisy, s=20, label='Observations')
plt.plot(X, y_true, 'k--', label='True model')
plt.xlabel('x'); plt.ylabel('y')
plt.title('Synthetic data')
plt.legend(); plt.tight_layout(); plt.show()

### Setting up `SteadyStateModel`

`SteadyStateModel` wraps a function `h(params) -> array` and exposes:
- `solve(time)` → `(0, 0, y)`  ← time is unused, kept for API compatibility
- `residuals(y_exp, t=0)` → `y_exp - y_model`
- `set_unknown_params(p_list)` → declare which parameters to fit and their bounds
- `estimate(y_exp, t=0, ...)` → run differential-evolution optimisation

In [ ]:
# Start from a deliberately wrong initial guess
model = mrpd.SteadyStateModel(model_fn, {'a': 1.0, 'b': 0.5})

# Declare which parameters to estimate:
#   (name, (lower_bound, upper_bound), is_linear_scale, display_label)
model.set_unknown_params([
    ('a', (0., 8.), True, 'a'),
    ('b', (0., 5.), True, 'b'),
])

print("Initial residuals RMS:", np.sqrt(np.mean(model.residuals(y_noisy, t=0)**2)))

In [ ]:
sol, p_est = model.estimate(
    y_noisy, t=0,
    print_iterations=False,
    popsize=20,
    ftol=1e-10,
    penalty_threshold=0,
)

print(f"Estimated  a = {p_est[0]:.4f}  (true = {TRUE_PARAMS['a']})")
print(f"Estimated  b = {p_est[1]:.4f}  (true = {TRUE_PARAMS['b']})")
print(f"Optimiser status: {sol.message}")

In [ ]:
y_fit = model_fn(model.p)

plt.figure(figsize=(5, 3))
plt.scatter(X, y_noisy, s=20, label='Observations')
plt.plot(X, y_true, 'k--', lw=1.5, label='True model')
plt.plot(X, y_fit, 'C1-', lw=2, label='Fitted model')
plt.xlabel('x'); plt.ylabel('y')
plt.title('Parameter estimation result')
plt.legend(); plt.tight_layout(); plt.show()

## Part 2 — Fitting a PEMFC polarization curve

We generate noisy synthetic "experimental" data from a known cell, then recover
the exchange current density `i0_ref` and charge-transfer coefficient `alpha` by
fitting the same model structure.

### 2.1 Build the reference cell and generate data

In [ ]:
def build_cell(i0_ref=2.5e-4, alpha=0.5):
    liq = mrpd.DarcyTransportModel(J_function_exponent=2)
    orr = mrpd.ElectrochemicalReaction(
        reference_exchange_current_density=i0_ref,
        reaction_order=0.54,
        activation_energy=67e6,
        reference_activity=1e5,
        reference_temperature=353.15,
        number_of_electrons=2,
        charge_transfer_coeff=alpha,
    )
    cl_ca = mrpd.PtCCatalystLayer(
        ecsa=70e3, platinum_loading=0.4e-2, ionomer=mrpd.PFSAIonomer(),
        reaction=orr, thickness=10e-6, thermal_conductivity=0.22,
        pore_diameter=40e-9, absolute_permeability=1e-13, contact_angle=97.,
        two_phase_transport_model=liq,
    )
    gdl = mrpd.GasDiffusionLayer(
        thickness=200e-6, porosity=0.6, contact_angle=120.,
        effective_gas_diffusion_ratio=0.3, absolute_permeability=1e-12,
        thermal_conductivity=0.5, two_phase_transport_model=liq,
    )
    def ch(r): return mrpd.FlowChannel(width=1e-3, height=1e-3, length=0.1, n_parallel=20, reactant=r)
    return mrpd.FuelCell(
        area=25e-4, electrical_resistance=30e-7,
        ca=mrpd.FuelCellSide(cl=cl_ca, gdl=gdl, ch=ch('o2'), has_mpl=False, thermal_contact_resistance=4e-4),
        an=mrpd.FuelCellSide(
            cl=mrpd.PtCCatalystLayer(thickness=5e-6, two_phase_transport_model=liq),
            gdl=mrpd.GasDiffusionLayer(
                thickness=200e-6, porosity=0.6, contact_angle=120.,
                effective_gas_diffusion_ratio=0.3, absolute_permeability=1e-12,
                thermal_conductivity=0.5, two_phase_transport_model=liq,
            ),
            ch=ch('h2'), has_mpl=False, thermal_contact_resistance=4e-4,
        ),
        membrane=mrpd.PFSA(
            ionomer=mrpd.PFSAIonomer(equivalent_weight=1100, dry_density=1980),
            dry_thickness=25e-6,
        ),
        use_eq_water_content_for_ionomer=True,
    )


T = 353.15

def make_conditions(i_k):
    """Return a CellConditions for a single current density point."""
    return mrpd.CellConditions(
        current_density=np.array([float(i_k)]),
        cell_temperature=T,
        ca=mrpd.SideConditions(
            inlet_temperature=T, inlet_pressure=1.5e5, outlet_pressure=1.5e5,
            dry_o2_mole_fraction=0.21, inlet_relative_humidity=0.5, stoichiometry=2.0,
        ),
        an=mrpd.SideConditions(
            inlet_temperature=T, inlet_pressure=1.5e5, outlet_pressure=1.5e5,
            dry_h2_mole_fraction=1.0, inlet_relative_humidity=0.5, stoichiometry=1.5,
        ),
    )


# True parameters
I0_TRUE   = 2.5e-4   # A/m²_Pt
ALPHA_TRUE = 0.5

ref_cell = build_cell(i0_ref=I0_TRUE, alpha=ALPHA_TRUE)
ref_model = mrpd.ExplicitSteadyStateModel()

i_Acm2 = np.array([0.1, 0.3, 0.5, 0.8, 1.0, 1.2, 1.5, 1.8, 2.0])
i_Am2  = i_Acm2 * 1e4

V_ref = []
for i_k in i_Am2:
    cond = make_conditions(i_k)
    state = ref_model.set_initial_conditions(ref_cell, cond)
    state = ref_model.solve(ref_cell, cond, state)
    V_ref.append(float(np.atleast_1d(state.cell_voltage)[0]))
V_ref = np.array(V_ref)

# Add synthetic measurement noise
V_exp = V_ref + rng.normal(0, 3e-3, size=V_ref.shape)

plt.figure(figsize=(5, 3))
plt.plot(i_Acm2, V_ref, 'k--', label='True model')
plt.scatter(i_Acm2, V_exp, s=30, zorder=5, label='"Measured" (+ noise)')
plt.xlabel('Current density (A/cm²)'); plt.ylabel('Cell voltage (V)')
plt.legend(); plt.tight_layout(); plt.show()

### 2.2 Define the estimation model

The model function takes a parameter dict, builds a cell with those values, and
returns the simulated voltage at each current density.

We fit two kinetic parameters:
- `i0` — reference exchange current density (log-scale)
- `alpha` — charge-transfer coefficient (linear scale)

In [ ]:
def polarization_model(params):
    cell = build_cell(
        i0_ref=10**params['log_i0'],
        alpha=params['alpha'],
    )
    model = mrpd.ExplicitSteadyStateModel()
    voltages = []
    for i_k in i_Am2:
        cond = make_conditions(i_k)
        state = model.set_initial_conditions(cell, cond)
        state = model.solve(cell, cond, state)
        voltages.append(float(np.atleast_1d(state.cell_voltage)[0]))
    return np.array(voltages)


# Initial guess — deliberately off from truth
init_params = {'log_i0': np.log10(1e-4), 'alpha': 0.4}

estimator = mrpd.SteadyStateModel(polarization_model, init_params)

In [ ]:
# Declare parameters to estimate
# Tuple format: (param_name, (lower_bound, upper_bound), is_linear_scale, display_label)
# is_linear_scale=False means the parameter is searched on a log scale internally.
estimator.set_unknown_params([
    ('log_i0', (-6., -2.), True, r'$\log_{10}(i_0)$'),
    ('alpha',  (0.2, 0.8), True, r'$\alpha$'),
])

print("Fitting...")
sol, p_est = estimator.estimate(
    V_exp, t=0,
    print_iterations=False,
    popsize=15,
    ftol=1e-9,
    penalty_threshold=0,
)

print(f"\nEstimated  i0    = {10**p_est[0]:.3e}  (true = {I0_TRUE:.3e} A/m²_Pt)")
print(f"Estimated  alpha = {p_est[1]:.4f}    (true = {ALPHA_TRUE})")
print(f"Optimiser status: {sol.message}")

In [ ]:
V_fit = polarization_model(estimator.p)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

ax = axes[0]
ax.scatter(i_Acm2, V_exp, s=30, zorder=5, label='"Measured"')
ax.plot(i_Acm2, V_ref, 'k--', lw=1.5, label='True model')
ax.plot(i_Acm2, V_fit, 'C1-', lw=2, label='Fitted model')
ax.set_xlabel('Current density (A/cm²)')
ax.set_ylabel('Cell voltage (V)')
ax.set_title('Polarization curve fit')
ax.legend()

ax = axes[1]
residuals = V_exp - V_fit
ax.bar(i_Acm2, residuals * 1e3, width=0.12, color='C0')
ax.axhline(0, color='k', lw=0.8)
ax.set_xlabel('Current density (A/cm²)')
ax.set_ylabel('Residual (mV)')
ax.set_title('Fit residuals')

fig.tight_layout()
plt.show()

rmse_mV = np.sqrt(np.mean(residuals**2)) * 1e3
print(f"RMSE = {rmse_mV:.2f} mV")

## Summary

| | API call |
|---|---|
| Define model | `mrpd.SteadyStateModel(h, params)` |
| Declare unknowns | `model.set_unknown_params([(name, bounds, linear, label), ...])` |
| Fit | `sol, p_est = model.estimate(y_exp, t=0, ...)` |
| Residuals | `model.residuals(y_exp, t=0)` |

The optimiser (`differential_evolution` from SciPy) is global and derivative-free,
which makes it robust to non-convex cost surfaces that arise in electrochemical
models.